# 01 — Python Data Structures for Data Engineers

### Learning objectives

- choose appropriate Python structures for common data tasks;
- transform records without hiding the transformation logic;
- use functions and comprehensions responsibly;
- validate assumptions with assertions;
- measure a simple operation;
- recognize when a Python solution should eventually move into SQL or a distributed engine.

This is deliberately **not** a general Python course. We learn the Python that will support later data-system experiments.


## Capability contract

**Capability:** C01 — System Foundations  
**Workstream:** Foundation  
**Primary roles:** Data Engineer|Analytics Engineer  
**You will prove:** Explain the mechanism and reproduce it with a deterministic experiment.


## 🖊️ Sketch note

![Sketch note — Python Data Structures](../assets/sketch-notes/01-python-data-structures.svg)

> **Visual-first rule:** spend 30–60 seconds explaining the sketch in your own words before reading the implementation. Then return to it after the experiment and redraw it from memory.


## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

This foundation notebook does not have one dedicated Acme Commerce ticket. It is a **cross-cutting prerequisite** used throughout the apprenticeship. Use it whenever a scenario requires the underlying runtime, Python or basic data-handling skill.

**Scenario map:** see [APPRENTICESHIP_MAP.md](../APPRENTICESHIP_MAP.md).


## Why this exists

Start with the engineering problem. Know what would go wrong if this mechanism did not exist.


## Predict before you run

Predict which structure will make the target operation cheapest or clearest, and explain why.


## Diagnose

When the experiment behaves differently from your prediction, inspect the operation and representation before changing code.


In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import date
from typing import Iterable

events = [
    {"event_id": 1, "customer_id": "C1", "event": "purchase", "amount": 120.0},
    {"event_id": 2, "customer_id": "C2", "event": "view",     "amount": 0.0},
    {"event_id": 3, "customer_id": "C1", "event": "purchase", "amount": 80.0},
    {"event_id": 4, "customer_id": "C3", "event": "purchase", "amount": 55.0},
    {"event_id": 5, "customer_id": "C1", "event": "view",     "amount": 0.0},
]

print(events[0])


## 1. Choose the structure for the question

A list is good when order matters.

A dictionary is good for key → value lookup.

A `Counter` is useful for frequencies.

A `defaultdict` is useful when accumulating grouped values.

The engineering question is not:

> "Which Python feature do I know?"

It is:

> "What data access pattern do I need?"


In [ ]:
event_counts = Counter(e["event"] for e in events)

revenue_by_customer = defaultdict(float)
for e in events:
    if e["event"] == "purchase":
        revenue_by_customer[e["customer_id"]] += e["amount"]

print("Event counts:", event_counts)
print("Revenue:", dict(revenue_by_customer))


In [ ]:
# Turn the transformation into a small, testable function.
def purchases(events: Iterable[dict]) -> list[dict]:
    return [e for e in events if e["event"] == "purchase"]

purchase_rows = purchases(events)

assert len(purchase_rows) == 3
assert all(e["event"] == "purchase" for e in purchase_rows)

print(purchase_rows)
print("Function test: PASS")


## 2. Data classes: useful, but don't over-engineer

Once a record has a stable structure, a dataclass can make assumptions explicit.

This is especially useful for reusable application code. It is not automatically better than dictionaries for exploratory work.


In [ ]:
@dataclass(frozen=True)
class Event:
    event_id: int
    customer_id: str
    event: str
    amount: float

typed_events = [Event(**e) for e in events]

assert typed_events[0].event == "purchase"
assert typed_events[0].amount == 120.0

typed_events[:2]


## 3. Measure before optimizing

For a tiny list, almost any approach is fast enough.

That is an important engineering lesson: **optimization without measurement is usually guesswork.**

We will use a deliberately simple benchmark next.


In [ ]:
import time

large_events = events * 20_000

start = time.perf_counter()
purchase_rows = [e for e in large_events if e["event"] == "purchase"]
elapsed = time.perf_counter() - start

print(f"Rows processed: {len(large_events):,}")
print(f"Purchases: {len(purchase_rows):,}")
print(f"Elapsed: {elapsed:.4f} s")


## 4. Break it

What if an event arrives without `amount`?

Our current code assumes the field exists.

In production, malformed records are normal. Later modules will introduce schemas, validation, quarantine and data contracts.

For now, make the failure explicit.


In [ ]:
bad_events = events + [
    {"event_id": 6, "customer_id": "C4", "event": "purchase"}
]

def validate_event(event: dict) -> None:
    required = {"event_id", "customer_id", "event", "amount"}
    missing = required - event.keys()
    if missing:
        raise ValueError(f"Missing required fields: {sorted(missing)}")

for event in bad_events:
    try:
        validate_event(event)
    except ValueError as exc:
        print("Rejected:", exc)


## Engineering challenge

Write a function:

```python
summarize_customer_activity(events)
```

It should return, for each customer:

- number of events;
- number of purchases;
- total purchase revenue.

Add at least three assertions.

**Production connection:** later, the same transformation will become a SQL query, a batch transformation, or a distributed computation.
